# Skin Masking

Per-image **face-skin mask from RGB**, used to isolate the erythema target. **Approach:** MediaPipe
multiclass selfie segmenter, keeping the face-skin class (per-pixel; hair and background excluded by
class). **Contract:** target = destriped EI × mask.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config

PROJECT_ROOT  = Path(config.__file__).resolve().parent
PROCESSED     = (PROJECT_ROOT / config.LOCAL_PROCESSED_DIR).resolve()
DESTRIPED_DIR = PROCESSED / "ei_maps_destriped"          # median-destriped target (src.destripe_ei)
assert DESTRIPED_DIR.exists(), f"Destriped EI dir not found: {DESTRIPED_DIR}"

manifest = pd.read_csv(PROCESSED / "manifest.csv")
print(f"{len(manifest)} images   target dir: {DESTRIPED_DIR}")

## The mask — MediaPipe face-skin segmenter

Load MediaPipe's multiclass selfie segmenter once and define `compute_mask(rgb)` = its **face-skin
class** (per-pixel; hair and background excluded by class). No thresholding or morphology.

In [ ]:
# MediaPipe multiclass selfie segmenter (downloaded once into ../models, gitignored).
import urllib.request
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import vision, BaseOptions
from src.io_utils import load_rgb

SEG_MODEL = PROJECT_ROOT / "models" / "selfie_multiclass.tflite"
SEG_URL   = ("https://storage.googleapis.com/mediapipe-models/image_segmenter/"
             "selfie_multiclass_256x256/float32/latest/selfie_multiclass_256x256.tflite")
if not SEG_MODEL.exists():
    SEG_MODEL.parent.mkdir(exist_ok=True)
    print("downloading selfie_multiclass.tflite ...")
    urllib.request.urlretrieve(SEG_URL, SEG_MODEL)

_segmenter = vision.ImageSegmenter.create_from_options(
    vision.ImageSegmenterOptions(
        base_options=BaseOptions(model_asset_path=str(SEG_MODEL)),
        output_category_mask=True))

FACE_SKIN_CLASS = 3   # 0 bg, 1 hair, 2 body-skin, 3 face-skin, 4 clothes, 5 accessories

def compute_mask(rgb):
    """Binary face-skin mask from RGB (MediaPipe multiclass segmenter, class 3)."""
    img = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
    cat = _segmenter.segment(img).category_mask.numpy_view()
    if cat.ndim == 3:
        cat = cat[..., 0]
    return (cat == FACE_SKIN_CLASS).astype(np.uint8)

print("segmenter ready:", SEG_MODEL.name)

### Face-skin mask on every neutral image, per view

For all subjects, per neutral view (front / left / right): mask **outline** (green) and **segmented**
face (RGB × mask) side by side, ordered by split.

In [ ]:
import cv2

path_of = {f"{r.subject_id}_{r.pose}_{r.view}": r.rgb_path for r in manifest.itertuples(index=False)}

def show_view(view, subj_per_row=3):
    """Per subject, side by side: mask OUTLINE on RGB (left) and SEGMENTED face RGB×mask
    (right), for every neutral image of one view, ordered by split."""
    order = {"test": 0, "train": 1, "valid": 2}
    sub = manifest[(manifest.pose == "neutral") & (manifest.view == view)].copy()
    sub = sub.sort_values(by=["split", "subject_id"],
                          key=lambda c: c.map(order) if c.name == "split" else c)
    n = len(sub); ncol = subj_per_row * 2; nrow = int(np.ceil(n / subj_per_row))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.7*ncol, 3.0*nrow))
    axes = np.atleast_2d(axes).reshape(nrow, ncol)
    for k, r in enumerate(sub.itertuples(index=False)):
        rr, cc = k // subj_per_row, (k % subj_per_row) * 2
        rgb  = load_rgb(str(r.rgb_path)); mask = compute_mask(rgb)
        overlay = rgb.copy()
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, cnts, -1, (0, 255, 0), 3)
        axes[rr, cc].imshow(overlay)
        axes[rr, cc].set_title(f"{r.split[:2]} {r.subject_id}  outline  {mask.mean()*100:.0f}%", fontsize=8)
        axes[rr, cc].axis("off")
        axes[rr, cc+1].imshow(rgb/255.0 * mask[:, :, None])            # segmented = RGB × mask
        axes[rr, cc+1].set_title("segmented (RGB×mask)", fontsize=8)
        axes[rr, cc+1].axis("off")
    for k in range(n, nrow*subj_per_row):
        rr, cc = k // subj_per_row, (k % subj_per_row) * 2
        axes[rr, cc].axis("off"); axes[rr, cc+1].axis("off")
    plt.suptitle(f"Neutral — {view.upper()} — outline + segmented face-skin (all subjects), ordered by split",
                 y=1.002, fontsize=12)
    plt.tight_layout(); plt.show()

#### Neutral — FRONT (all subjects)

In [ ]:
show_view("front")

#### Neutral — LEFT (all subjects)

In [ ]:
show_view("left")

#### Neutral — RIGHT (all subjects)

In [ ]:
show_view("right")

## Validation over all 306 images

Coverage sanity — a plausible, stable face fraction with no runaway (lit backdrop) or collapse
(missed face) — in every split.

In [ ]:
rows = []
for r in manifest.itertuples(index=False):
    stem = f"{r.subject_id}_{r.pose}_{r.view}"
    mask = compute_mask(load_rgb(str(r.rgb_path)))
    rows.append(dict(img=stem, subject=r.subject_id, split=r.split,
                     view=r.view, skin_pct=mask.mean()*100))
res = pd.DataFrame(rows)

print("coverage % — overall:")
print(res.skin_pct.describe().round(1).to_string())
print("\ncoverage % by view (profiles should match frontal):")
print(res.groupby('view').skin_pct.describe()[['mean', 'min', 'max']].round(1).to_string())
print("\ncoverage % by split:")
print(res.groupby('split').skin_pct.describe()[['mean', 'min', 'max']].round(1).to_string())
print(f"\nzero masks: {(res.skin_pct == 0).sum()}   "
      f"runaway (>40%): {(res.skin_pct > 40).sum()}   "
      f"collapsed (<5%): {(res.skin_pct < 5).sum()}")

## Target close-up — disclosure subjects, all profiles

The three disclosure subjects (the only ones permitted for display), all three neutral profiles:
**mask × RGB** (masked input) and **mask × destriped EI** (the target), pixel-aligned.

In [ ]:
DISCLOSURE = ["p012", "p019", "p027"]
VIEWS = ["front", "left", "right"]
pairs = [(s, v) for s in DISCLOSURE for v in VIEWS]

fig, axes = plt.subplots(len(pairs), 3, figsize=(13, 4.2*len(pairs)))
for row, (s, v) in enumerate(pairs):
    stem = f"{s}_neutral_{v}"
    rgb  = load_rgb(str(path_of[stem]))
    d    = np.load(DESTRIPED_DIR / f"{stem}.npy")   # precomputed destriped target (src.destripe_ei)
    mask = compute_mask(rgb).astype(bool)
    mask_rgb = rgb/255.0 * mask[:, :, None]
    mask_ei  = np.where(mask, d, np.nan)             # mask × destriped EI = the target
    for ax, (im, t, cm, kw) in zip(axes[row], [
        (rgb,      f"{stem} — RGB", None, {}),
        (mask_rgb, "mask × RGB (input)", None, {}),
        (mask_ei,  "mask × EI (target)", "RdYlGn_r", dict(vmin=-50, vmax=100))]):
        img = ax.imshow(im, cmap=cm, **kw); ax.set_title(t, fontsize=10); ax.axis("off")
        if cm == "RdYlGn_r":
            plt.colorbar(img, ax=ax, fraction=0.046)
plt.suptitle("Disclosure subjects, all profiles — mask × RGB vs mask × destriped EI", y=1.003)
plt.tight_layout(); plt.show()

## Conclusion

**Mask = MediaPipe multiclass segmenter, face-skin class (RGB).** Full coverage at any pose, hair
excluded, background ignored. Over 306 images: no zero masks, no runaway, profiles match frontal. No
threshold, morphology, landmarks, or fallback. Ears and lips kept (valid skin; can't be removed
uniformly). **Contract: target = destriped EI × mask.**

**Next:** `compute_mask` → `src/masking.py` + `scripts/compute_masks.py`.